# 04 Baselines And Component Checks

Generated by `scripts/revision/create_revision_notebooks.py`.


## Setup

In [ ]:
from pathlib import Path
import json
import os
import subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = DRIVE_ROOT / 'ECG-RAMBA'
LOCAL_RUNTIME_ROOT = Path('/content/ecg_ramba_runtime')

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ.setdefault('ECG_RAMBA_LOCAL_ROOT', str(LOCAL_RUNTIME_ROOT))
os.environ.setdefault('ECG_RAMBA_EXTRACT_DIR', str(LOCAL_RUNTIME_ROOT / 'chapman'))
os.environ.setdefault('ECG_RAMBA_USE_CLEAN_CACHE', '0')
os.environ.setdefault('ECG_RAMBA_SAVE_CLEAN_CACHE', '0')

def _run_setup(cmd, cwd=None, check=True):
    print(f'$ {cmd}', flush=True)
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

def _git_setup(cmd, check=True):
    return _run_setup(cmd, cwd=REPO_DIR, check=check)

def _git_current_commit():
    result = _git_setup('git rev-parse --short HEAD', check=False)
    return result.stdout.strip() if result.returncode == 0 and result.stdout else 'unknown'

def _pull_or_continue(branch):
    before = _git_current_commit()
    status_result = _git_setup('git status --porcelain', check=False)
    status = status_result.stdout.strip() if status_result.stdout else ''
    if status:
        print('Local repo has changes before pull:')
        print(status[:4000])
    result = _git_setup(f'git pull --ff-only --autostash origin {branch}', check=False)
    after = _git_current_commit()
    if result.returncode:
        print('')
        print('=' * 80)
        print('WARNING: git pull failed; continuing with the currently checked-out repo.')
        print('This avoids deleting Drive files while long-running artifacts may exist.')
        print(f'Current commit: {after} (before pull: {before})')
        print('To force a clean code checkout later, rename the Drive repo folder or use a fresh clone.')
        print('=' * 80)
    else:
        print(f'Git update OK: {before} -> {after}')

if (REPO_DIR / '.git').exists():
    os.chdir(REPO_DIR)
    fetch_result = _git_setup('git fetch origin', check=False)
    if fetch_result.returncode:
        print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
    current_branch_result = _git_setup('git branch --show-current', check=False)
    current_branch = current_branch_result.stdout.strip() if current_branch_result.stdout else ''
    if current_branch != BRANCH:
        checkout_result = _git_setup(f'git checkout {BRANCH}', check=False)
        if checkout_result.returncode:
            print(f'WARNING: git checkout {BRANCH} failed; continuing on branch {current_branch or "<detached>"}')
        else:
            current_branch = BRANCH
    if fetch_result.returncode == 0:
        _pull_or_continue(BRANCH)
elif (REPO_DIR / 'configs' / 'config.py').exists():
    os.chdir(REPO_DIR)
    print('Repo directory exists but is not a git checkout; skipping git pull.')
elif Path.cwd().joinpath('configs', 'config.py').exists():
    REPO_DIR = Path.cwd()
    os.chdir(REPO_DIR)
    if (REPO_DIR / '.git').exists():
        fetch_result = _run_setup('git fetch origin', check=False)
        if fetch_result.returncode == 0:
            _pull_or_continue(BRANCH)
        else:
            print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
else:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    _run_setup(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)

os.chdir(REPO_DIR)
_run_setup('git status --short --branch', check=False)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)

cache_status = {
    'clean_raw_cache': DRIVE_ROOT / 'ecg_data_27c_subject.npz',
    'raw_minirocket_cache': DRIVE_ROOT / 'rocket_raw_N44186_C12_L5000_K10000_S42.npz',
    'hrv36_cache': DRIVE_ROOT / 'hrv36_N44186_C12_L5000.npz',
    'fold_pca_cache_dir': DRIVE_ROOT / 'revision_feature_cache',
}
print('cache policy:')
print('  ECG_RAMBA_USE_CLEAN_CACHE =', os.environ.get('ECG_RAMBA_USE_CLEAN_CACHE'))
print('  ECG_RAMBA_SAVE_CLEAN_CACHE=', os.environ.get('ECG_RAMBA_SAVE_CLEAN_CACHE'))
print('  ECG_RAMBA_EXTRACT_DIR     =', os.environ.get('ECG_RAMBA_EXTRACT_DIR'))
print('cache status:')
for name, path in cache_status.items():
    if path.is_dir():
        count = len(list(path.glob('*.npz')))
        print(f'  {name}: exists=True npz_count={count} path={path}')
    else:
        print(f'  {name}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')

def run(cmd, check=True, log_path=None, tail_lines=160):
    import time

    print(f'$ {cmd}', flush=True)
    if log_path is None:
        log_dir = Path('reports/revision/logs')
        log_dir.mkdir(parents=True, exist_ok=True)
        stamp = time.strftime('%Y%m%d_%H%M%S')
        log_path = log_dir / f'notebook_command_{stamp}.log'
    else:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open('w', encoding='utf-8') as log_file:
        proc = subprocess.Popen(
            cmd,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        return_code = proc.wait()

    print(f'Command log: {log_path}')
    if check and return_code:
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print(f'Command failed with exit code {return_code}. Last {min(tail_lines, len(lines))} log lines:')
        for line in lines[-tail_lines:]:
            print(line)
        raise subprocess.CalledProcessError(return_code, cmd)
    return subprocess.CompletedProcess(cmd, return_code)


def run_script_if_exists(script_path, command):
    path = Path(script_path)
    if path.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')


## Minimum Fair Baseline Matrix

A-level baselines: Full ECG-RAMBA, Raw Mamba, ResNet1D/CNN, MiniRocket-only, HRV-only. All must use the same split, preprocessing, threshold, and aggregation protocol.

In [ ]:
import pandas as pd

registry = pd.read_csv('docs/revision_plan/experiment_registry.csv')
display(registry[registry['experiment_id'].isin(['EXP_BASE', 'EXP_PRED'])])

planned_outputs = [
    'reports/revision/metrics/baseline_summary.csv',
    'reports/revision/tables/table_baselines.csv',
]
for path in planned_outputs:
    print(path, 'exists=', Path(path).exists())


## Placeholder Commands

In [ ]:
RUN_HEAVY = False

planned = {
    'MiniRocket-only': 'python scripts/revision/TBD_minirocket_only.py',
    'HRV-only': 'python scripts/revision/TBD_hrv_only.py',
    'ResNet1D/CNN': 'python scripts/revision/TBD_resnet1d_baseline.py',
    'Raw Mamba': 'python scripts/revision/TBD_raw_mamba_baseline.py',
}

for name, command in planned.items():
    if RUN_HEAVY:
        run(command, check=False)
    else:
        print(f'{name}: planned command -> {command}')
